# Stage 01 — Pipeline Fluency: Noisy Circle

Explores how noise and persistence thresholding affect TDA features on a noisy circle.

**Three-layer module structure**

| Module | Responsibility |
|---|---|
| `tda.tda_data` | Pure computation — generate, compute, filter, stats |
| `tda.tda_viz` | Atomic ax-based renderers — `render_*` functions |
| `tda.tda_figures` | GridSpec orchestrators — `plot_*` functions |

Each section below demonstrates the public API for that layer.

In [ ]:
import sys
sys.path.insert(0, '..')

from tda import (
    # data layer
    generate_point_cloud, compute_rips, filter_diagrams, h1_stats,
    # renderer layer
    render_point_cloud, render_persistence_diagram, render_barcode, render_landscape,
    # figure layer
    plot_noise_experiment,
)
import matplotlib.pyplot as plt

## Layer 1 — `tda_data`: Computation

Pure computation, no matplotlib. Returns typed dataclasses so you can inspect intermediate results.

| Function | Returns |
|---|---|
| `generate_point_cloud(shape, n_points, noise)` | `ndarray` — `'circle'` \| `'torus'` \| `'sphere'` |
| `compute_rips(pts, max_dim)` | `RipsResult` — `.dgms`, `.num_edges`, `.elapsed_ms` |
| `filter_diagrams(dgms, threshold)` | filtered diagram list; always keeps ∞ bars |
| `h1_stats(dgms)` | `(bar_count, max_persistence)` |

In [ ]:
# generate_point_cloud — 'circle' | 'torus' | 'sphere'
pts = generate_point_cloud('circle', n_points=100, noise=0.1)
print(f"pts shape: {pts.shape}")

# compute_rips — returns a RipsResult dataclass (no tuple unpacking)
rips = compute_rips(pts, max_dim=1)
print(f"edges: {rips.num_edges}  |  time: {rips.elapsed_ms:.1f}ms")
print(f"dgms[0] (H0) shape: {rips.dgms[0].shape}")
print(f"dgms[1] (H1) shape: {rips.dgms[1].shape}")

# filter_diagrams — drops finite bars with persistence ≤ threshold, always keeps ∞ bars
filtered = filter_diagrams(rips.dgms, threshold=0.05)
print(f"\nAfter filter(0.05) — H1 bars: {len(rips.dgms[1])} → {len(filtered[1])}")

# h1_stats — quick summary of the H1 diagram
count, max_pers = h1_stats(rips.dgms)
print(f"h1_stats: {count} finite bars, max persistence = {max_pers:.4f}")

## Layer 2 — `tda_viz`: Renderers

Atomic ax-based renderers. Each takes `(data, ax, **kwargs)` and returns `None`.
The caller owns the figure layout — no `plt.show()` inside any renderer.

| Function | Key kwargs |
|---|---|
| `render_point_cloud(pts, ax)` | `title` |
| `render_persistence_diagram(dgms, ax)` | `title` |
| `render_barcode(dgms, ax)` | `title`, `dims` — e.g. `(1,)` for H₁ only, `(0,1)` for H₀+H₁ |
| `render_landscape(dgms, ax)` | `title`, `dims`, `n_landscapes` |

In [ ]:
pts  = generate_point_cloud('circle', n_points=100, noise=0.1)
rips = compute_rips(pts, max_dim=1)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# render_point_cloud — scatter for 2D or 3D point clouds
render_point_cloud(pts, axes[0], title="Point Cloud")

# render_persistence_diagram — wraps persim.plot_diagrams; handles inf bars
render_persistence_diagram(rips.dgms, axes[1], title="Persistence Diagram")

# render_barcode — dims=(1,) → H₁ only; dims=(0,1) → H₀ and H₁ together
# infinite bars shown with arrow tips
render_barcode(rips.dgms, axes[2], title="H₁ Barcode", dims=(1,))

# render_landscape — tent-function computation (no external library)
# n_landscapes controls how many λₖ curves to draw per dimension
render_landscape(rips.dgms, axes[3], title="H₁ Landscape", dims=(1,), n_landscapes=3)

plt.tight_layout()
plt.show()

## Layer 3 — `plot_noise_experiment`: Figure Orchestrator

Combines a point cloud with any selection of diagram representations into a multi-panel figure.

```python
plot_noise_experiment(
    pts, rips, noise,
    thresholds,                           # float  →  one row of diagrams
                                          # [unfilt, filt]  →  two comparison rows
    representations = ('pd', 'bc', 'pl'), # columns: any ordered subset
    mode = 'h1only',                      # 'h1only' | 'overlay' | 'grid'
)
```

**`mode` values**

| Mode | Effect on barcode / landscape columns |
|---|---|
| `'h1only'` | H₁ only |
| `'overlay'` | H₀ + H₁ on one axis |
| `'grid'` | 2×2 sub-grid: H₁ \| H₀+H₁  ×  unfiltered \| filtered |

In [ ]:
# Single threshold — one row, all three representations side by side
# Noise sweep shows how the dominant H1 bar grows while noise bars accumulate
for noise in [0.0, 0.05, 0.15, 0.3, 0.5]:
    pts  = generate_point_cloud('circle', n_points=100, noise=noise)
    rips = compute_rips(pts, max_dim=1)
    plot_noise_experiment(
        pts, rips, noise,
        thresholds=0.0,
        representations=['pd', 'bc', 'pl'],
        mode='h1only',
    )

In [ ]:
# Two thresholds [unfiltered, filtered] — two comparison rows
# mode='grid' shows H₁ and H₀+H₁ × unfiltered and filtered in a 2×2 sub-grid
configs = [
    (100, 0.05, [0.0, 0.05]),
    (100, 0.15, [0.0, 0.05]),
    (100, 0.3,  [0.0, 0.1]),
    (100, 0.5,  [0.0, 0.15]),
]
for n, noise, thresholds in configs:
    pts  = generate_point_cloud('circle', n_points=n, noise=noise)
    rips = compute_rips(pts, max_dim=1)
    plot_noise_experiment(
        pts, rips, noise,
        thresholds=thresholds,
        representations=['bc', 'pl'],
        mode='grid',
    )

In [ ]:
# mode='overlay' — H₀ + H₁ on the same axis
# Useful for seeing connected-component collapse alongside loop formation
pts  = generate_point_cloud('circle', n_points=100, noise=0.2)
rips = compute_rips(pts, max_dim=1)
plot_noise_experiment(
    pts, rips, noise=0.2,
    thresholds=[0.0, 0.08],
    representations=['bc', 'pl'],
    mode='overlay',
)